# Dual-Branch AI Image Detector — Kaggle Version

## Setup Datasets yang Dibutuhkan

Sebelum run notebook ini, tambahkan dataset berikut di Kaggle:

### 1. Dataset Publik (Add via `+ Add Data`)
| Dataset | Kaggle Slug | Isi |
|---|---|---|
| GenImage-BigGAN | `yangtao2021/imagenet-ai-0419-biggan` | 162k AI + 162k Nature |
| GenImage-GLide | `yangtao2021/imagenet-glide` | 162k AI + 162k Nature |

### 2. Dataset Private (Upload sendiri)
Upload folder `Gemini/` Anda ke Kaggle:
- Pergi ke **kaggle.com/datasets** → **New Dataset**
- Upload semua 2086 file JPG
- Beri nama: `gemini-ai-images`
- Set ke **Private**
- Lalu **Add Data** ke notebook ini

### 3. Aktifkan GPU
Settings → Accelerator → **GPU T4 x2** (gratis)


In [ ]:
import os

# Auto-detect Kaggle dataset paths
KAGGLE_INPUT = '/kaggle/input'

# Cari dataset yang sudah di-add
print('Available Kaggle datasets:')
if os.path.exists(KAGGLE_INPUT):
    for d in os.listdir(KAGGLE_INPUT):
        full = os.path.join(KAGGLE_INPUT, d)
        count = sum(len(files) for _, _, files in os.walk(full))
        print(f'  /kaggle/input/{d}/  ({count:,} files)')
else:
    print('  Not running on Kaggle!')


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models, transforms
from torch.utils.data import DataLoader, Dataset, random_split, ConcatDataset, Subset
from torchvision.datasets import ImageFolder
import torch.fft
import numpy as np
import os, random, shutil
from pathlib import Path

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')


In [ ]:
# ============================================================
# KONFIGURASI DATASET
# Dataset: Unbiased Tiny GenImage (atau dataset serupa)
# Struktur: /kaggle/input/<slug>/BigGAN/, glide/, Nature/
# ============================================================

KAGGLE_INPUT = '/kaggle/input'

# Auto-detect dataset slug
def find_dataset_root(keywords):
    for d in os.listdir(KAGGLE_INPUT):
        for kw in keywords:
            if kw.lower() in d.lower():
                return os.path.join(KAGGLE_INPUT, d)
    return None

GENIMAGE_ROOT = find_dataset_root(['genimage', 'tiny', 'unbiased'])
if GENIMAGE_ROOT is None:
    GENIMAGE_ROOT = os.path.join(KAGGLE_INPUT, os.listdir(KAGGLE_INPUT)[0])
    print('WARNING: Auto-detect failed, using first dataset:', GENIMAGE_ROOT)
else:
    print('Dataset root:', GENIMAGE_ROOT)

print('Contents:', os.listdir(GENIMAGE_ROOT))

# Paths ke folder generator (flat, langsung berisi gambar)
BIGGAN_DIR = os.path.join(GENIMAGE_ROOT, 'BigGAN')
GLIDE_DIR  = os.path.join(GENIMAGE_ROOT, 'glide')
NATURE_DIR = os.path.join(GENIMAGE_ROOT, 'Nature')

# Gemini: set path jika sudah upload, else None untuk skip
GEMINI_DIR = None  # Contoh: '/kaggle/input/gemini-ai-images'

# Kuota gambar per sumber
BIGGAN_QUOTA = 4000
GLIDE_QUOTA  = 4000
GEMINI_QUOTA = 2000  # diabaikan jika GEMINI_DIR = None

# Working dir
WORKING    = '/kaggle/working'
MERGED_DIR = os.path.join(WORKING, 'merged_dataset')
MODEL_PATH = os.path.join(WORKING, 'best_detector_model.pth')

def dir_count(p):
    return len(os.listdir(p)) if os.path.exists(p) else 0

print('BigGAN exists:', os.path.exists(BIGGAN_DIR), '|', dir_count(BIGGAN_DIR), 'files')
print('GLide  exists:', os.path.exists(GLIDE_DIR),  '|', dir_count(GLIDE_DIR),  'files')
print('Nature exists:', os.path.exists(NATURE_DIR), '|', dir_count(NATURE_DIR), 'files')
print('Gemini set   :', GEMINI_DIR is not None)


In [ ]:
# Build merged_dataset/ from multiple sources
# (Idempotent: skip jika sudah dikopi)

IMG_EXTS = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}

def collect(folder):
    if not os.path.exists(folder):
        print('  WARNING: not found ->', folder)
        return []
    files = [os.path.join(folder, f) for f in os.listdir(folder)
             if os.path.splitext(f)[1].lower() in IMG_EXTS]
    print('  Found', len(files), 'images in', os.path.basename(folder))
    return files

def copy_sample(imgs, dest_dir, n, prefix):
    os.makedirs(dest_dir, exist_ok=True)
    existing = len([f for f in os.listdir(dest_dir)
                    if os.path.isfile(os.path.join(dest_dir, f))])
    if existing >= n:
        print('  [' + prefix + '] Already done (' + str(existing) + ' files), skipping.')
        return existing
    take = min(n - existing, len(imgs))
    if take < (n - existing):
        print('  [' + prefix + '] WARNING: only', len(imgs), 'available')
    chosen = random.sample(imgs, take)
    for i, src in enumerate(chosen):
        ext = os.path.splitext(src)[1].lower()
        shutil.copy2(src, os.path.join(dest_dir,
                     prefix + '_' + str(existing + i).zfill(6) + ext))
    print('  [' + prefix + '] Copied', take)
    return existing + take

ai_dest     = os.path.join(MERGED_DIR, 'ai')
nature_dest = os.path.join(MERGED_DIR, 'nature')

# --- AI class: BigGAN + GLide + Gemini (opsional) ---
print('--- Building AI class ---')
total_ai = 0
ai_sources = [
    (BIGGAN_DIR, BIGGAN_QUOTA, 'biggan'),
    (GLIDE_DIR,  GLIDE_QUOTA,  'glide'),
]
if GEMINI_DIR and os.path.exists(GEMINI_DIR):
    ai_sources.append((GEMINI_DIR, GEMINI_QUOTA, 'gemini'))
else:
    print('  Gemini: skipped (GEMINI_DIR not set)')

for src, quota, prefix in ai_sources:
    imgs = collect(src)
    total_ai += copy_sample(imgs, ai_dest, quota, prefix)

# --- Nature class: dari NATURE_DIR (flat folder) ---
print('\n--- Building Nature class ---')
all_nature = collect(NATURE_DIR)
copy_sample(all_nature, nature_dest, total_ai, 'nature')

ai_n  = len([f for f in os.listdir(ai_dest)     if os.path.isfile(os.path.join(ai_dest, f))])
nat_n = len([f for f in os.listdir(nature_dest) if os.path.isfile(os.path.join(nature_dest, f))])
print('\nMerged dataset ready:', ai_n, 'AI |', nat_n, 'Nature | Total', ai_n + nat_n)
print('Balance:', 'OK' if ai_n == nat_n else 'UNBALANCED')


In [ ]:
# 1. Spatial Branch (CNN) - ConvNeXt
class SpatialBranch(nn.Module):
    def __init__(self, backbone_name='convnext_tiny'):
        super(SpatialBranch, self).__init__()
        if backbone_name == 'efficientnet_b4':
            self.backbone = models.efficientnet_b4(weights=models.EfficientNet_B4_Weights.DEFAULT)
            self.feature_extractor = nn.Sequential(*list(self.backbone.children())[:-1])
            self.feature_dim = 1792
            self._backbone_type = 'standard'
        elif backbone_name == 'resnet50':
            self.backbone = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
            self.feature_extractor = nn.Sequential(*list(self.backbone.children())[:-1])
            self.feature_dim = 2048
            self._backbone_type = 'standard'
        elif backbone_name == 'convnext_base':
            _b = models.convnext_base(weights=models.ConvNeXt_Base_Weights.IMAGENET1K_V1)
            self.feature_extractor = _b.features
            self.avgpool = _b.avgpool
            self.feature_dim = 1024
            self._backbone_type = 'convnext'
        elif backbone_name == 'convnext_small':
            _b = models.convnext_small(weights=models.ConvNeXt_Small_Weights.IMAGENET1K_V1)
            self.feature_extractor = _b.features
            self.avgpool = _b.avgpool
            self.feature_dim = 768
            self._backbone_type = 'convnext'
        elif backbone_name == 'convnext_tiny':
            _b = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.IMAGENET1K_V1)
            self.feature_extractor = _b.features
            self.avgpool = _b.avgpool
            self.feature_dim = 768
            self._backbone_type = 'convnext'
        else:
            raise ValueError(f'Unknown backbone: {backbone_name}')

        self.artifact_attention = nn.Sequential(
            nn.Linear(self.feature_dim, self.feature_dim // 4),
            nn.ReLU(),
            nn.Linear(self.feature_dim // 4, self.feature_dim),
            nn.Sigmoid()
        )
        print(f'[SpatialBranch] Backbone: {backbone_name} | feature_dim: {self.feature_dim}')

    def forward(self, x):
        if self._backbone_type == 'convnext':
            x = self.feature_extractor(x)
            x = self.avgpool(x)
            features = x.flatten(1)
        else:
            features = self.feature_extractor(x)
            features = features.view(features.size(0), -1)
        attention_weights = self.artifact_attention(features)
        return features * attention_weights


In [ ]:
# 2. Frequency Branch (2D-DFT)
class FrequencyBranch(nn.Module):
    def __init__(self):
        super(FrequencyBranch, self).__init__()
        self.freq_analyzer = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1))
        )
        self.feature_dim = 32

    def forward(self, x):
        fft_result  = torch.fft.fft2(x)
        fft_shifted = torch.fft.fftshift(fft_result)
        magnitude   = torch.abs(fft_shifted)
        log_magnitude = torch.log1p(magnitude)
        freq_features = self.freq_analyzer(log_magnitude)
        return freq_features.view(freq_features.size(0), -1)


In [ ]:
# 3. Dual-Branch Detector
class DualBranchDetector(nn.Module):
    def __init__(self, backbone_name='convnext_tiny'):
        super(DualBranchDetector, self).__init__()
        self.spatial_branch   = SpatialBranch(backbone_name)
        self.frequency_branch = FrequencyBranch()
        fusion_dim = self.spatial_branch.feature_dim + self.frequency_branch.feature_dim
        self.classifier = nn.Sequential(
            nn.Linear(fusion_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, 1)
        )

    def forward(self, x):
        spatial_feat = self.spatial_branch(x)
        freq_feat    = self.frequency_branch(x)
        fused        = torch.cat([spatial_feat, freq_feat], dim=1)
        return self.classifier(fused)


In [ ]:
# 4. DataLoaders - Optimized for Kaggle GPU
def get_dataloaders(data_dir, batch_size=32, subset_per_class=None):
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

    full_dataset = ImageFolder(root=data_dir, transform=transform)
    print(f'Full dataset: {len(full_dataset)} images')
    print(f'Classes: {full_dataset.classes}')

    # Optional subset for quick experiments
    if subset_per_class is not None:
        targets  = np.array(full_dataset.targets)
        indices  = []
        for cls_idx in range(len(full_dataset.classes)):
            cls_indices = np.where(targets == cls_idx)[0]
            chosen = np.random.choice(cls_indices,
                                      min(subset_per_class, len(cls_indices)),
                                      replace=False)
            indices.extend(chosen.tolist())
        np.random.shuffle(indices)
        dataset = Subset(full_dataset, indices)
        print(f'Subset: {len(dataset)} images ({subset_per_class}/class)')
    else:
        dataset = full_dataset
        print(f'Using FULL dataset: {len(dataset)} images')

    total      = len(dataset)
    train_size = int(0.7 * total)
    val_size   = int(0.15 * total)
    test_size  = total - train_size - val_size

    generator = torch.Generator().manual_seed(42)
    train_ds, val_ds, test_ds = random_split(dataset, [train_size, val_size, test_size], generator=generator)
    print(f'Split: {train_size} Train | {val_size} Val | {test_size} Test')

    # num_workers=4 optimal for Kaggle (4 CPU cores available)
    kwargs = dict(num_workers=4, pin_memory=True)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,  **kwargs)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, **kwargs)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False, **kwargs)

    return train_loader, val_loader, test_loader


In [ ]:
# 5. Training Loop
def train_model(model, train_loader, val_loader, epochs=50, lr=1e-4, patience=5, device='cuda'):
    model     = model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.BCEWithLogitsLoss()
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

    best_val_loss    = float('inf')
    epochs_no_improve = 0

    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device).float()
            optimizer.zero_grad()
            outputs = model(inputs).squeeze()
            loss    = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        model.eval()
        val_loss, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device).float()
                outputs = model(inputs).squeeze()
                val_loss += criterion(outputs, labels).item()
                preds    = (torch.sigmoid(outputs) >= 0.5).int()
                correct += (preds == labels.int()).sum().item()
                total   += labels.size(0)

        avg_train = train_loss / len(train_loader)
        avg_val   = val_loss   / len(val_loader)
        val_acc   = correct / total
        scheduler.step(avg_val)

        print(f'Epoch {epoch+1}/{epochs} | Train Loss: {avg_train:.4f} | '
              f'Val Loss: {avg_val:.4f} | Val Acc: {val_acc:.4f}')

        if avg_val < best_val_loss:
            best_val_loss = avg_val
            epochs_no_improve = 0
            torch.save(model.state_dict(), MODEL_PATH)
            print(f'  >> Model saved to {MODEL_PATH}')
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f'Early stopping after {patience} epochs without improvement.')
                break

    print('Training complete. Best val loss:', round(best_val_loss, 4))
    return model


In [ ]:
# 6. Execution
# Dataset loading - full 20k (no subset needed on Kaggle GPU)
train_loader, val_loader, test_loader = get_dataloaders(
    MERGED_DIR,
    batch_size=64,          # Kaggle T4 bisa handle batch 64
    subset_per_class=None,  # None = pakai full dataset
)

# Model - gunakan convnext_base di Kaggle karena VRAM lebih besar (16GB)
model = DualBranchDetector(backbone_name='convnext_base')

# Training
train_model(model, train_loader, val_loader, epochs=50, lr=1e-4, patience=5, device=device)


In [ ]:
# 7. Evaluation
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

def evaluate_model(model, test_loader, device='cuda'):
    model.load_state_dict(torch.load(MODEL_PATH))
    model = model.to(device)
    model.eval()
    all_preds, all_labels = [], []

    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs = inputs.to(device)
            outputs = model(inputs)
            preds   = (torch.sigmoid(outputs).squeeze() >= 0.5).int().cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())

    print('=== Test Set Results ===')
    print(classification_report(all_labels, all_preds, target_names=['Nature/Real', 'AI-Generated']))
    print('Accuracy :', round(accuracy_score(all_labels, all_preds), 4))
    print('Precision:', round(precision_score(all_labels, all_preds), 4))
    print('Recall   :', round(recall_score(all_labels, all_preds), 4))
    print('F1 Score :', round(f1_score(all_labels, all_preds), 4))

evaluate_model(model, test_loader, device=device)
